In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

# 1 Load Data

In [ ]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
data_path_goodreads = base_artifacts / 'Datasets' / 'Processed' / 'goodreads'
data_path_sequels = base_artifacts / 'Datasets' / 'Sequels'

data = pd.read_csv(data_path_goodreads / 'data_clean.csv')
with open(data_path_goodreads / 'item_dict.pkl', 'rb') as f:
    item_dict = pickle.load(f)
n_items = len(item_dict)
title2id = {title: item_id for item_id, title in item_dict.items()}

with open(data_path_sequels / 'name2series.pkl', 'rb') as f:
    name2series = pickle.load(f)

id2series = {
    item_id: name2series[title]['series'] 
    for item_id, title in item_dict.items() 
    if title in name2series
}
id2author = lambda item_id: item_dict[item_id].split(' by ')[1]

# 2 Find Good Series

Find series from which there is are at least two books present in the data

In [3]:
df = data.copy()

# Map item_id to series
df['series'] = df['item_id'].map(id2series)
df['author'] = df['item_id'].map(id2author)

# Drop books that don't have a series
df = df.dropna().reset_index(drop=True)

# Find series with more than 1 item
ser = df.groupby(['author', 'series'])['item_id'].nunique()
chosen_series = ser[ser > 1].index.tolist()
chosen_titles = [title for title in name2series if name2series[title]['series'] in chosen_series]

# Find the item_ids of the chosen series
unique_ids = data['item_id'].unique()
present_titles = [item_dict[item_id] for item_id in unique_ids]
chosen_titles = [
    title 
    for title in present_titles 
    if title in name2series
    and (id2author(title2id[title]), name2series[title]['series']) in chosen_series
]
chosen_ids = [title2id[title] for title in chosen_titles]

print(f'Number of series with at least 2 items: {len(chosen_series):,} (out of {len(ser):,})')
print(f'Number of titles in chosen series: {len(chosen_titles):,} (out of {len(present_titles):,})')

Number of series with at least 2 items: 735 (out of 1,954)
Number of titles in chosen series: 2,561 (out of 6,384)


# 3 Build Ground Truth 

In [4]:
results = {'title_A': [], 'title_B': [], 'causal_link': []}
chosen_pairs_ids = []
chosen_pairs_titles = []
for item1 in chosen_ids:
    for item2 in chosen_ids:
        if id2series[item1] != id2series[item2]:
            continue
        if id2author(item1) != id2author(item2):
            continue

        title1 = item_dict[item1]
        title2 = item_dict[item2]
        num1 = name2series[title1]['number']
        num2 = name2series[title2]['number']
        if num1 == num2:
            continue

        if num1 < num2:
            link = 1
        else:
            link = 0

        results['title_A'].append(title1)
        results['title_B'].append(title2)
        results['causal_link'].append(link)
        chosen_pairs_ids.append((item1, item2))
        chosen_pairs_titles.append((title1, title2))

oracle = pd.DataFrame(results)

# 4 Save Results

In [5]:
oracle.to_csv(data_path_sequels / 'oracle.csv', index=False)

chosen_pairs_dir = base_artifacts / 'Chosen_Pairs' / 'sequels'
with open(chosen_pairs_dir / 'chosen_pairs_ids.pkl', 'wb') as f:
    pickle.dump(chosen_pairs_ids, f)
with open(chosen_pairs_dir / 'chosen_pairs_titles.pkl', 'wb') as f:
    pickle.dump(chosen_pairs_titles, f)